# 03 – Kiểm soát nhiễu và huấn luyện có nhận thức địa lý (trên đặc trưng XLS-R đã cache)

**Phần 1 – Mô hình có học giọng hay học địa danh, kênh thu?**
- So khớp tên tỉnh trong transcript, không phân biệt kiểu đặt dấu (hòa/hoà).
- Huấn luyện lại TF-IDF trên văn bản đã che địa danh.
- Tính accuracy của XLS-R trên các tập con: không nhắc địa danh nào; mô hình văn bản đoán sai; LTAS đoán sai.

**Phần 2 – Đóng góp chính:** cùng head weighted-sum + MLP trên XLS-R, thay đổi hàm mục tiêu:
`CE` | `Label smoothing` (đối chứng) | `SORD` (soft label theo khoảng cách địa lý, τ km) | `Đa nhiệm vùng miền` | `Đa nhiệm tọa độ`.
Mỗi cấu hình chạy 3 seed. Tham số được chọn theo **valid**.

**Đánh giá:** accuracy, macro-F1, top-5, đúng vùng, **khoảng cách sai trung bình (km)** và tỉ lệ đoán trong bán kính 100 km.

- **Runtime:** GPU L4 hoặc T4, khoảng 20–30 phút.
- **Yêu cầu:** đã chạy notebook 01 và 02.
- **Kết quả:** `results/geo_report.json` và 3 hình `.png`. Gửi cho Claude file JSON và 3 hình.

In [ ]:
# ==== CẤU HÌNH ====
DRIVE_ROOT = "/content/drive/MyDrive/vimd_dl"
PROVINCE_COL, TEXT_COL = "province_code", "text"
SEEDS = [0, 1, 2]
SORD_TAUS = [50, 100, 200, 400]        # km
LS_EPS = [0.1, 0.2]
REGION_LAMBDAS = [0.3, 1.0]
COORD_LAMBDAS = [0.3, 1.0]

In [ ]:
import os, re, glob, json, time, unicodedata, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score, top_k_accuracy_score
warnings.filterwarnings("ignore")
T0 = time.time()
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
FEAT_DIR, RES_DIR = f"{DRIVE_ROOT}/features", f"{DRIVE_ROOT}/results"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiết bị:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

meta = pd.read_csv(f"{DRIVE_ROOT}/metadata.csv", low_memory=False)
meta[PROVINCE_COL] = meta[PROVINCE_COL].astype(str)
prov = pd.read_csv(f"{DRIVE_ROOT}/provinces.csv"); prov[PROVINCE_COL] = prov[PROVINCE_COL].astype(str)
CLASSES = sorted(meta[PROVINCE_COL].unique()); C = len(CLASSES)
y = meta[PROVINCE_COL].map({c: i for i, c in enumerate(CLASSES)}).values
use = ~meta["exclude"].astype(bool).values
tr, va, te = [(meta["split"].values == s) & use for s in ("train", "valid", "test")]
code2name = dict(zip(prov[PROVINCE_COL], prov["province_name"]))
NAMES = [code2name[c] for c in CLASSES]
region_of = np.array([prov.set_index(PROVINCE_COL).loc[c, "region"] for c in CLASSES])
print(f"{C} tỉnh | train {tr.sum()} | valid {va.sum()} | test {te.sum()}")

## Bảng tỉnh lỵ: tên tiếng Việt, tọa độ, địa danh phụ (63 tỉnh theo địa giới trước 2025)
Tọa độ là của trung tâm tỉnh lỵ, sai số khoảng ±5 km. Mức sai số này không đáng kể khi khoảng cách giữa các tỉnh tính bằng hàng chục đến hàng trăm km.

In [ ]:
GEO = {  # key: tên trong provinces.csv -> (tên tiếng Việt, vĩ độ, kinh độ, [địa danh phụ])
 "CaoBang": ("Cao Bằng", 22.666, 106.258, []), "LangSon": ("Lạng Sơn", 21.853, 106.761, []),
 "QuangNinh": ("Quảng Ninh", 20.951, 107.080, ["Hạ Long", "Cẩm Phả", "Móng Cái"]),
 "HaiPhong": ("Hải Phòng", 20.845, 106.688, []), "ThaiBinh": ("Thái Bình", 20.446, 106.336, []),
 "NamDinh": ("Nam Định", 20.420, 106.168, []), "PhuTho": ("Phú Thọ", 21.322, 105.402, ["Việt Trì"]),
 "ThaiNguyen": ("Thái Nguyên", 21.594, 105.848, []), "YenBai": ("Yên Bái", 21.705, 104.875, []),
 "TuyenQuang": ("Tuyên Quang", 21.823, 105.214, []), "HaGiang": ("Hà Giang", 22.823, 104.984, []),
 "LaoCai": ("Lào Cai", 22.486, 103.971, ["Sa Pa"]), "LaiChau": ("Lai Châu", 22.396, 103.458, []),
 "SonLa": ("Sơn La", 21.326, 103.917, []), "DienBien": ("Điện Biên", 21.386, 103.023, []),
 "HoaBinh": ("Hòa Bình", 20.817, 105.338, []), "HaNoi": ("Hà Nội", 21.028, 105.854, []),
 "HaiDuong": ("Hải Dương", 20.940, 106.333, []), "NinhBinh": ("Ninh Bình", 20.251, 105.975, []),
 "VinhPhuc": ("Vĩnh Phúc", 21.310, 105.597, ["Vĩnh Yên"]), "HungYen": ("Hưng Yên", 20.646, 106.051, []),
 "HaNam": ("Hà Nam", 20.541, 105.914, ["Phủ Lý"]), "BacKan": ("Bắc Kạn", 22.147, 105.835, ["Bắc Cạn"]),
 "BacGiang": ("Bắc Giang", 21.273, 106.195, []), "BacNinh": ("Bắc Ninh", 21.186, 106.076, []),
 "ThanhHoa": ("Thanh Hóa", 19.807, 105.776, []), "NgheAn": ("Nghệ An", 18.679, 105.682, []),
 "HaTinh": ("Hà Tĩnh", 18.343, 105.906, []), "QuangBinh": ("Quảng Bình", 17.468, 106.622, ["Đồng Hới"]),
 "QuangTri": ("Quảng Trị", 16.816, 107.100, ["Đông Hà"]), "ThuaThienHue": ("Thừa Thiên Huế", 16.464, 107.586, ["Huế"]),
 "DaNang": ("Đà Nẵng", 16.054, 108.202, []), "QuangNam": ("Quảng Nam", 15.573, 108.474, ["Tam Kỳ", "Hội An"]),
 "QuangNgai": ("Quảng Ngãi", 15.120, 108.792, []), "BinhDinh": ("Bình Định", 13.776, 109.224, ["Quy Nhơn"]),
 "PhuYen": ("Phú Yên", 13.096, 109.309, ["Tuy Hòa"]), "KhanhHoa": ("Khánh Hòa", 12.239, 109.197, ["Nha Trang", "Cam Ranh"]),
 "NinhThuan": ("Ninh Thuận", 11.565, 108.988, ["Phan Rang"]), "BinhThuan": ("Bình Thuận", 10.928, 108.102, ["Phan Thiết"]),
 "KonTum": ("Kon Tum", 14.350, 108.000, []), "GiaLai": ("Gia Lai", 13.983, 108.000, ["Pleiku", "Plây Cu"]),
 "DakLak": ("Đắk Lắk", 12.667, 108.038, ["Buôn Ma Thuột", "Đắc Lắc"]), "DakNong": ("Đắk Nông", 12.004, 107.690, ["Gia Nghĩa"]),
 "LamDong": ("Lâm Đồng", 11.940, 108.458, ["Đà Lạt"]), "HoChiMinh": ("Hồ Chí Minh", 10.776, 106.701, ["Sài Gòn"]),
 "DongNai": ("Đồng Nai", 10.957, 106.843, ["Biên Hòa"]), "BinhDuong": ("Bình Dương", 10.980, 106.651, ["Thủ Dầu Một"]),
 "BaRiaVungTau": ("Bà Rịa", 10.496, 107.169, ["Vũng Tàu"]), "TayNinh": ("Tây Ninh", 11.310, 106.098, []),
 "BinhPhuoc": ("Bình Phước", 11.535, 106.883, ["Đồng Xoài"]), "LongAn": ("Long An", 10.535, 106.413, []),
 "TienGiang": ("Tiền Giang", 10.360, 106.360, ["Mỹ Tho"]), "BenTre": ("Bến Tre", 10.241, 106.376, []),
 "VinhLong": ("Vĩnh Long", 10.254, 105.972, []), "TraVinh": ("Trà Vinh", 9.935, 106.345, []),
 "CanTho": ("Cần Thơ", 10.045, 105.747, []), "HauGiang": ("Hậu Giang", 9.784, 105.470, ["Vị Thanh"]),
 "SocTrang": ("Sóc Trăng", 9.603, 105.980, []), "BacLieu": ("Bạc Liêu", 9.294, 105.727, []),
 "CaMau": ("Cà Mau", 9.176, 105.150, []), "DongThap": ("Đồng Tháp", 10.460, 105.633, ["Cao Lãnh", "Sa Đéc"]),
 "AnGiang": ("An Giang", 10.386, 105.435, ["Long Xuyên", "Châu Đốc"]), "KienGiang": ("Kiên Giang", 10.012, 105.081, ["Rạch Giá", "Phú Quốc"]),
}
missing = [n for n in NAMES if n not in GEO]
assert not missing, f"Thiếu tọa độ cho: {missing}"
LAT = np.array([GEO[n][1] for n in NAMES]); LON = np.array([GEO[n][2] for n in NAMES])

def haversine(lat1, lon1, lat2, lon2):
    p = np.pi / 180
    a = np.sin((lat2 - lat1) * p / 2) ** 2 + np.cos(lat1 * p) * np.cos(lat2 * p) * np.sin((lon2 - lon1) * p / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))

DIST = haversine(LAT[:, None], LON[:, None], LAT[None], LON[None])   # (C, C) km
nn_km = np.sort(DIST + np.eye(C) * 1e9, 1)[:, 0]
print(f"Khoảng cách tới tỉnh gần nhất: trung vị {np.median(nn_km):.0f} km, max {nn_km.max():.0f} km")

## Phần 1. Kiểm soát nhiễu nội dung & kênh thu

In [ ]:
TONE_FIX = {"oà": "òa", "oá": "óa", "oả": "ỏa", "oã": "õa", "oạ": "ọa", "oè": "òe", "oé": "óe", "oẻ": "ỏe",
            "oẽ": "õe", "oẹ": "ọe", "uỳ": "ùy", "uý": "úy", "uỷ": "ủy", "uỹ": "ũy", "uỵ": "ụy"}

def norm(s):
    s = unicodedata.normalize("NFC", str(s).lower())
    for a, b in TONE_FIX.items():
        s = s.replace(a, b)
    return re.sub(r"\s+", " ", s)

place_terms = {k: [norm(GEO[n][0])] + [norm(a) for a in GEO[n][3]] for k, n in enumerate(NAMES)}
all_terms = sorted({t for ts in place_terms.values() for t in ts}, key=len, reverse=True)
PLACE_RE = re.compile(r"(?<!\w)(" + "|".join(map(re.escape, all_terms)) + r")(?!\w)")
own_re = {k: re.compile(r"(?<!\w)(" + "|".join(map(re.escape, ts)) + r")(?!\w)") for k, ts in place_terms.items()}

texts = np.array([norm(t) for t in meta[TEXT_COL].fillna("")])
mentions_any = np.array([bool(PLACE_RE.search(t)) for t in texts])
mentions_own = np.array([bool(own_re[k].search(t)) for k, t in zip(y, texts)])
LEAK = dict(own_rate_all=mentions_own.mean(), any_rate_all=mentions_any.mean(),
            own_rate_test=mentions_own[te].mean(), any_rate_test=mentions_any[te].mean())
LEAK = {k: round(float(v), 4) for k, v in LEAK.items()}
print("Tỉ lệ câu nhắc địa danh:", LEAK)

# TF-IDF trên văn bản đã che địa danh (so với kết quả 02 chưa che)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from scipy.sparse import hstack, csr_matrix
masked = np.array([PLACE_RE.sub(" ", t) for t in texts])
vw = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(masked[tr])
vc = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True, max_features=300_000).fit(masked[tr])
Xm = csr_matrix(hstack([vw.transform(masked), vc.transform(masked)]))
best = max(((LinearSVC(C=c).fit(Xm[tr], y[tr]), c) for c in (0.1, 0.5, 1.0)),
           key=lambda m: (m[0].predict(Xm[va]) == y[va]).mean())[0]
pred_text_masked = best.predict(Xm[te])
LEAK["text_tfidf_masked_acc"] = round(float((pred_text_masked == y[te]).mean()), 4)
print("TF-IDF trên văn bản đã che địa danh, acc test:", LEAK["text_tfidf_masked_acc"])

In [ ]:
PRED = {os.path.basename(f)[5:-4]: np.load(f) for f in glob.glob(f"{RES_DIR}/pred_*.npy")}
PRED["text_tfidf_masked"] = pred_text_masked
yt = y[te]
rng = np.random.default_rng(0)

def ci(ok, n_boot=1000):
    if len(ok) == 0:
        return [np.nan, np.nan]
    m = rng.integers(0, len(ok), (n_boot, len(ok)))
    return np.percentile(ok[m].mean(1), [2.5, 97.5]).round(4).tolist()

subsets = {
    "all": np.ones(len(yt), bool),
    "no_place_name": ~mentions_any[te],
    "mentions_own_province": mentions_own[te],
}
for k in ("text_tfidf", "acoustic_ltas", "acoustic_mfcc"):
    if k in PRED:
        subsets[f"{k}_wrong"] = PRED[k] != yt
if "text_tfidf" in PRED and "acoustic_ltas" in PRED:
    subsets["text_and_ltas_wrong"] = (PRED["text_tfidf"] != yt) & (PRED["acoustic_ltas"] != yt)

models = [m for m in ("xlsr_wsum_mlp", "xlsr_best_layer_linear", "acoustic_mfcc", "acoustic_ltas",
                      "text_tfidf", "text_tfidf_masked") if m in PRED]
rows = []
for sname, m in subsets.items():
    r = dict(subset=sname, n=int(m.sum()))
    for k in models:
        r[k] = round(float((PRED[k][m] == yt[m]).mean()), 4) if m.any() else np.nan
    r["xlsr_wsum_mlp_ci95"] = ci((PRED["xlsr_wsum_mlp"][m] == yt[m]).astype(float)) if "xlsr_wsum_mlp" in PRED else None
    rows.append(r)
SUBSET_TABLE = pd.DataFrame(rows)
print(SUBSET_TABLE.to_string(index=False))

## Phần 2. Huấn luyện có nhận thức địa lý trên đặc trưng XLS-R đóng băng

In [ ]:
fdir = sorted(glob.glob(f"{FEAT_DIR}/xlsr_*"))[0]
print("Dùng đặc trưng:", fdir)
GROUPS = {s: g.index.values for s, g in meta.groupby("shard")}
first = np.load(f"{fdir}/{next(iter(GROUPS))}.npy")
XL = np.zeros((len(meta),) + first.shape[1:], np.float16)
for s, idx in GROUPS.items():
    XL[idx] = np.load(f"{fdir}/{s}.npy")
Xg = torch.tensor(XL, device=device); del XL
mtr = torch.tensor(tr, device=device)
mu, sd = Xg[mtr].float().mean(0), Xg[mtr].float().std(0) + 1e-5
for i in range(0, len(Xg), 2048):
    Xg[i:i + 2048] = ((Xg[i:i + 2048].float() - mu) / sd).half()
Xtr_, Xva_, Xte_ = [Xg[torch.tensor(m, device=device)] for m in (tr, va, te)]
del Xg
nL, D = Xtr_.shape[1], Xtr_.shape[2]
ytr_ = torch.tensor(y[tr], device=device)
REG = sorted(set(region_of)); reg_idx = torch.tensor([REG.index(r) for r in region_of], device=device)
coord = np.stack([(LAT - LAT.mean()) / LAT.std(), (LON - LON.mean()) / LON.std()], 1)
coord_t = torch.tensor(coord, dtype=torch.float32, device=device)
DIST_t = torch.tensor(DIST, dtype=torch.float32, device=device)

class GeoHead(nn.Module):
    def __init__(s):
        super().__init__(); s.w = nn.Parameter(torch.zeros(nL))
        s.body = nn.Sequential(nn.Linear(D, 512), nn.GELU(), nn.Dropout(0.3))
        s.prov, s.reg, s.xy = nn.Linear(512, C), nn.Linear(512, len(REG)), nn.Linear(512, 2)
    def forward(s, x):
        h = s.body((x.float() * s.w.softmax(0)[None, :, None]).sum(1))
        return s.prov(h), s.reg(h), s.xy(h)

def make_loss(kind, p):
    if kind == "ce":
        return lambda o, yb: F.cross_entropy(o[0], yb)
    if kind == "ls":
        return lambda o, yb: F.cross_entropy(o[0], yb, label_smoothing=p)
    if kind == "sord":
        Q = torch.softmax(-DIST_t / p, dim=1)          # hàng i: phân phối mục tiêu cho tỉnh i
        return lambda o, yb: -(Q[yb] * F.log_softmax(o[0], -1)).sum(1).mean()
    if kind == "region":
        return lambda o, yb: F.cross_entropy(o[0], yb) + p * F.cross_entropy(o[1], reg_idx[yb])
    if kind == "coord":
        return lambda o, yb: F.cross_entropy(o[0], yb) + p * F.mse_loss(o[2], coord_t[yb])
    raise ValueError(kind)

@torch.no_grad()
def predict(head, X, bs=2048):
    head.eval()
    return torch.cat([head(X[i:i + bs])[0].float().softmax(-1) for i in range(0, len(X), bs)]).cpu().numpy()

def train(kind, p, seed, epochs=80, lr=1e-3, wd=1e-3, bs=256, patience=10):
    torch.manual_seed(seed); np.random.seed(seed)
    head = GeoHead().to(device); opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
    loss_fn = make_loss(kind, p); best, state, bad = -1, None, 0
    for ep in range(epochs):
        head.train(); perm = torch.randperm(len(Xtr_), device=device)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            loss = loss_fn(head(Xtr_[b]), ytr_[b])
            opt.zero_grad(); loss.backward(); opt.step()
        acc = (predict(head, Xva_).argmax(1) == y[va]).mean()
        if acc > best:
            best, state, bad = acc, {k: v.clone() for k, v in head.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                break
    head.load_state_dict(state)
    return head

def geo_metrics(probs, yy):
    pred = probs.argmax(1); d = DIST[yy, pred]
    k = min(5, C - 1)
    return dict(acc=(pred == yy).mean(), macro_f1=f1_score(yy, pred, average="macro"),
                top5=top_k_accuracy_score(yy, probs, k=k, labels=np.arange(C)),
                region_acc=(region_of[pred] == region_of[yy]).mean(),
                mean_err_km=d.mean(), mean_err_km_wrong=d[pred != yy].mean() if (pred != yy).any() else 0.0,
                within_100km=(d <= 100).mean(), expected_km=(probs * DIST[yy]).sum(1).mean())

In [ ]:
GRID = [("ce", None)] + [("ls", e) for e in LS_EPS] + [("sord", t) for t in SORD_TAUS] \
       + [("region", l) for l in REGION_LAMBDAS] + [("coord", l) for l in COORD_LAMBDAS]
RUNS, TEST_PROBS = [], {}
for kind, p in GRID:
    t = time.time()
    for seed in SEEDS:
        head = train(kind, p, seed)
        mv = geo_metrics(predict(head, Xva_), y[va]); pt = predict(head, Xte_); mt = geo_metrics(pt, yt)
        RUNS.append(dict(kind=kind, param=np.nan if p is None else float(p), seed=seed, **{f"valid_{k}": float(v) for k, v in mv.items()},
                         **{f"test_{k}": float(v) for k, v in mt.items()}))
        TEST_PROBS[(kind, p, seed)] = pt
    sub = pd.DataFrame(RUNS[-len(SEEDS):])
    print(f"{kind:7s} {str(p):5s} | valid acc {sub.valid_acc.mean():.4f} | test acc {sub.test_acc.mean():.4f}"
          f"±{sub.test_acc.std():.4f} | err {sub.test_mean_err_km.mean():.0f} km | {time.time() - t:.0f}s", flush=True)

runs = pd.DataFrame(RUNS)
sel_runs = lambda k, pv: runs[(runs.kind == k) & (runs.param.isna() if pd.isna(pv) else np.isclose(runs.param, pv))]
# Mỗi phương pháp: chọn tham số có valid acc trung bình cao nhất
mean_valid = runs.groupby(["kind", "param"], dropna=False)["valid_acc"].mean().reset_index()
chosen = mean_valid.loc[mean_valid.groupby("kind")["valid_acc"].idxmax()]
cols = ["acc", "macro_f1", "top5", "region_acc", "mean_err_km", "mean_err_km_wrong", "within_100km", "expected_km"]
MAIN = []
for _, r in chosen.iterrows():
    s = sel_runs(r.kind, r.param)
    MAIN.append(dict(kind=r.kind, param=r.param, **{c: f"{s['test_' + c].mean():.4f} ± {s['test_' + c].std():.4f}" for c in cols}))
MAIN = pd.DataFrame(MAIN)
print(f"\nKết quả test (tham số chọn theo valid, trung bình ± độ lệch chuẩn qua {len(SEEDS)} seed):")
print(MAIN.to_string(index=False))

## So sánh cặp với CE (paired bootstrap, dùng xác suất trung bình của 3 seed)

In [ ]:
def ens(kind, p):
    return np.mean([TEST_PROBS[(kind, p, s)] for s in SEEDS], 0)

P_ce = ens("ce", None); pred_ce = P_ce.argmax(1)
B = rng.integers(0, len(yt), (2000, len(yt)))
COMPARE = {}
for _, r in chosen.iterrows():
    if r.kind == "ce":
        continue
    p = None if pd.isna(r.param) else r.param
    P = ens(r.kind, p); pr = P.argmax(1)
    dacc = (pr == yt).astype(float) - (pred_ce == yt)
    dkm = DIST[yt, pr] - DIST[yt, pred_ce]
    COMPARE[f"{r.kind}({p}) - ce"] = dict(
        acc_diff=round(float(dacc.mean()), 4), acc_ci95=np.percentile(dacc[B].mean(1), [2.5, 97.5]).round(4).tolist(),
        err_km_diff=round(float(dkm.mean()), 1), err_km_ci95=np.percentile(dkm[B].mean(1), [2.5, 97.5]).round(1).tolist())
print(json.dumps(COMPARE, indent=1))

# Lỗi của CE có tập trung vào tỉnh láng giềng không?
from scipy.stats import spearmanr
M = np.zeros((C, C))
np.add.at(M, (yt, pred_ce), 1)
off = ~np.eye(C, dtype=bool)
rho, pval = spearmanr(M[off], DIST[off])
wrong = pred_ce != yt
CONF = dict(spearman_conf_vs_dist=round(float(rho), 4), p_value=float(pval),
            mean_err_km_wrong=round(float(DIST[yt[wrong], pred_ce[wrong]].mean()), 1),
            random_wrong_km=round(float((DIST.sum(1) / (C - 1))[yt[wrong]].mean()), 1))
print("Nhầm lẫn và khoảng cách:", CONF)

## Hình minh họa (lưu vào `results/`)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#52514e", "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e",
                     "ytick.color": "#52514e", "axes.facecolor": "#fcfcfb", "figure.facecolor": "#fcfcfb",
                     "font.size": 9})
REG_STYLE = {"North": ("#2a78d6", "o", "Bắc"), "Central": ("#eb6834", "s", "Trung"), "South": ("#1baf7a", "^", "Nam")}
acc_prov = np.array([(pred_ce[yt == k] == k).mean() if (yt == k).any() else np.nan for k in range(C)])

# Hình 1: accuracy từng tỉnh theo vị trí địa lý
fig, ax = plt.subplots(figsize=(4.2, 6.4))
for rname, (col, mk, lab) in REG_STYLE.items():
    m = region_of == rname
    ax.scatter(LON[m], LAT[m], s=20 + 260 * np.nan_to_num(acc_prov[m]), c=col, marker=mk, alpha=0.85,
               edgecolors="#fcfcfb", linewidths=2, label=lab)
for k in np.argsort(acc_prov)[:5]:
    ax.annotate(f"{NAMES[k]} {acc_prov[k]:.0%}", (LON[k], LAT[k]), fontsize=7, color="#52514e",
                xytext=(4, -3), textcoords="offset points")
ax.set_xlabel("Kinh độ"); ax.set_ylabel("Vĩ độ"); ax.set_aspect("equal")
ax.set_title("Accuracy từng tỉnh (kích thước điểm) – CE", fontsize=9, loc="left")
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([], [], ls="", marker=mk, color=col, markersize=7, label=lab) for col, mk, lab in REG_STYLE.values()],
          frameon=False, loc="lower left", title="Vùng")
fig.tight_layout(); fig.savefig(f"{RES_DIR}/fig_province_acc_map.png"); plt.show()

# Hình 2: tần suất nhầm lẫn theo khoảng cách
fig, ax = plt.subplots(figsize=(5.2, 3.2))
bins = np.array([0, 100, 200, 400, 700, 1000, 1500, 2000])
cnt = np.histogram(DIST[off], bins, weights=M[off])[0]; pairs = np.histogram(DIST[off], bins)[0]
rate = cnt / np.maximum(pairs, 1)
ax.bar(range(len(rate)), rate, color="#2a78d6", width=0.8)
ax.set_xticks(range(len(rate)), [f"{a}–{b}" for a, b in zip(bins[:-1], bins[1:])], rotation=30)
ax.set_xlabel("Khoảng cách giữa tỉnh thật và tỉnh dự đoán (km)"); ax.set_ylabel("Số lần nhầm / cặp tỉnh")
ax.set_title(f"Lỗi tập trung ở tỉnh gần (Spearman ρ = {rho:.2f})", fontsize=9, loc="left")
ax.grid(axis="y", color="#e6e5e0", linewidth=0.6); ax.set_axisbelow(True)
fig.tight_layout(); fig.savefig(f"{RES_DIR}/fig_confusion_vs_distance.png"); plt.show()

# Hình 3: phân phối khoảng cách sai – CE so với SORD tốt nhất
sord_row = chosen[chosen.kind == "sord"]
if len(sord_row):
    tau = sord_row.param.iloc[0]; pred_s = ens("sord", tau).argmax(1)
    fig, ax = plt.subplots(figsize=(5.2, 3.2))
    for pr, col, lab in [(pred_ce, "#2a78d6", "CE"), (pred_s, "#eb6834", f"SORD τ={tau:g} km")]:
        d = np.sort(DIST[yt, pr]); ax.plot(d, np.arange(1, len(d) + 1) / len(d), color=col, linewidth=2, label=lab)
    ax.set_xlabel("Khoảng cách sai (km, 0 = đúng tỉnh)"); ax.set_ylabel("Tỉ lệ tích lũy câu test")
    ax.set_title("Phân phối khoảng cách sai", fontsize=9, loc="left")
    ax.grid(color="#e6e5e0", linewidth=0.6); ax.legend(frameon=False, loc="lower right")
    fig.tight_layout(); fig.savefig(f"{RES_DIR}/fig_error_distance_cdf.png"); plt.show()

In [ ]:
report = dict(leak=LEAK, subset_table=SUBSET_TABLE.to_dict(orient="records"),
              main=MAIN.to_dict(orient="records"), compare_vs_ce=COMPARE, confusion=CONF,
              all_runs=runs.round(4).to_dict(orient="records"),
              nearest_province_km=dict(median=round(float(np.median(nn_km)), 1), max=round(float(nn_km.max()), 1)),
              per_province_acc_ce={NAMES[k]: (None if np.isnan(a) else round(float(a), 4)) for k, a in enumerate(acc_prov)},
              gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
              total_minutes=round((time.time() - T0) / 60, 1))
json.dump(report, open(f"{RES_DIR}/geo_report.json", "w"), ensure_ascii=False, indent=1, default=str)
print("Đã lưu", f"{RES_DIR}/geo_report.json", "+ 3 hình .png – gửi cho Claude.")